# Concept of a Hyperplane in 2D — Spam vs Ham

Companion to `3bHyperplaneIn2D.md`.

A **hyperplane** is the boundary that separates classes. In 2D it is a **line**:

$$W_1 x_1 + W_2 x_2 + c = 0$$

We classify an email by the **sign** of $f(x)=W_1 x_1 + W_2 x_2 + c$.

## 1. Load data and pick two features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATH = 'Module Resources - SVM and Naive Bayes/SVM - Spam Classification/Spam.csv'
df = pd.read_csv(PATH)

# the two features the lecture uses
f1, f2 = 'word_freq_technology', 'word_freq_money'
X = df[[f1, f2]].values
y = df['spam'].values   # 1 = spam, 0 = ham
df[[f1, f2, 'spam']].head()

## 2. Draw a hyperplane (line) by hand

Pick coefficients $W_1, W_2, c$ ourselves and see the line + the two sides.
The line is just the set of points where $f(x)=0$.

In [ ]:
W1, W2, c = 2.0, 3.0, -1.0   # a hand-chosen line

def f(x1, x2):
    return W1*x1 + W2*x2 + c

# three sample emails from the worked example in the .md
for name, (x1, x2) in {'A':(0,1.0), 'B':(0.5,0), 'C':(0,0.1)}.items():
    val = f(x1, x2)
    side = 'on line' if abs(val)<1e-9 else ('positive' if val>0 else 'negative')
    print(f'Email {name}: f=({val:+.2f})  -> {side}')

In [ ]:
# x2 on the line for a given x1:  W1*x1 + W2*x2 + c = 0  =>  x2 = -(W1*x1 + c)/W2
xs = np.linspace(0, 4, 100)
line_x2 = -(W1*xs + c)/W2

ham  = y==0
spam = y==1
plt.figure(figsize=(7,6))
plt.scatter(X[ham,0],  X[ham,1],  s=8, alpha=.3, color='steelblue', label='ham (f>0 side)')
plt.scatter(X[spam,0], X[spam,1], s=8, alpha=.3, color='crimson',  label='spam (f<0 side)')
plt.plot(xs, line_x2, 'k-', lw=2, label='hyperplane: f(x)=0')
plt.xlabel(f1); plt.ylabel(f2)
plt.xlim(0,4); plt.ylim(0,4)
plt.title('A 2D hyperplane is just a line')
plt.legend(); plt.show()

## 3. The SIGN of f(x) is the prediction

Shade the plane by the sign of $f(x)$ — positive on one side, negative on the other.

In [ ]:
xx, yy = np.meshgrid(np.linspace(0,4,300), np.linspace(0,4,300))
Z = f(xx, yy)

plt.figure(figsize=(7,6))
plt.contourf(xx, yy, np.sign(Z), levels=[-2,0,2], alpha=.15, colors=['crimson','steelblue'])
plt.contour(xx, yy, Z, levels=[0], colors='k', linewidths=2)   # the hyperplane f=0
plt.scatter(X[ham,0],  X[ham,1],  s=8, alpha=.3, color='steelblue')
plt.scatter(X[spam,0], X[spam,1], s=8, alpha=.3, color='crimson')
plt.xlabel(f1); plt.ylabel(f2); plt.xlim(0,4); plt.ylim(0,4)
plt.title('sign(f) splits the plane: blue f>0, red f<0')
plt.show()

## 4. Let an SVM LEARN the W coefficients

Instead of guessing $W_1, W_2, c$, fit a linear SVM. The learned weights *are* the model.
(Standardise first — SVM is distance-based.)

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)
clf = SVC(kernel='linear', C=1.0).fit(Xs, y)

# In sklearn the decision function is  w·x + b ; note y=1 (spam) is the positive class here
w = clf.coef_[0]; b = clf.intercept_[0]
print(f'Learned hyperplane (standardised space):')
print(f'  W1 = {w[0]:+.3f}   W2 = {w[1]:+.3f}   c = {b:+.3f}')
print(f'  => f(x) = {w[0]:+.3f}*x1 {w[1]:+.3f}*x2 {b:+.3f}')
print('Training accuracy:', round(clf.score(Xs, y), 3))

In [ ]:
# Plot the LEARNED hyperplane
x_min,x_max = Xs[:,0].min()-.5, Xs[:,0].max()+.5
y_min,y_max = Xs[:,1].min()-.5, Xs[:,1].max()+.5
xx, yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7,6))
plt.contourf(xx, yy, np.sign(Z), levels=[-2,0,2], alpha=.12, colors=['steelblue','crimson'])
plt.contour(xx, yy, Z, levels=[0], colors='k', linewidths=2)  # learned hyperplane
plt.scatter(Xs[y==0,0], Xs[y==0,1], s=8, alpha=.3, color='steelblue', label='ham')
plt.scatter(Xs[y==1,0], Xs[y==1,1], s=8, alpha=.3, color='crimson',  label='spam')
plt.xlabel(f1+' (scaled)'); plt.ylabel(f2+' (scaled)')
plt.title('The SVM-learned hyperplane')
plt.legend(); plt.show()

## Summary

```
• A 2D hyperplane is a line:  W1*x1 + W2*x2 + c = 0
• sign(f(x)) gives the class:  f>0 one side, f<0 the other, f=0 on the line
• The W coefficients ARE the model — an SVM learns them from data.
```

**Next:** the same idea in 3D (a plane), then *which* hyperplane is best —
the maximum-margin one. See [[2SVMMotivation]] and [[1BiasVsVariance]].